# CSS Rating 2 — Qwen2.5-7B QLoRA (Colab Enterprise)
Headless. GPU 메모리 자동 감지 → batch/seq/lora_r 조정.

In [ ]:
!nvidia-smi

In [ ]:
import os, subprocess
BUCKET = 'css-rating2-training-wsd2605'
BASE = 'css-rating2'
os.makedirs('/tmp/work', exist_ok=True)
os.chdir('/tmp/work')
subprocess.run(f'gsutil cp gs://{BUCKET}/{BASE}/src.tar.gz . && tar -xzf src.tar.gz', shell=True, check=True)
subprocess.run(f'mkdir -p data/llm_seed && gsutil -m cp gs://{BUCKET}/{BASE}/data/llm_seed/*.jsonl data/llm_seed/', shell=True, check=True)
print('Source + data ready')

In [ ]:
# Remove torch_xla preloaded in container (if present), install known-good QLoRA stack
!pip uninstall -y torch_xla torch-xla 2>/dev/null || true
!pip install --no-cache-dir -q -U transformers==4.44.2 accelerate==0.33.0 peft==0.12.0 trl==0.9.6 bitsandbytes==0.43.3 datasets==2.21.0 sentencepiece einops scikit-learn
import torch
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0))
GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f'GPU memory: {GPU_MEM_GB:.1f} GB')

In [ ]:
# Auto-tune training config based on GPU memory
import torch, os
GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / (1024**3)
if GPU_MEM_GB > 35:
    BATCH, ACCUM, SEQ, LORA_R, LORA_A = 8, 2, 1024, 16, 32
elif GPU_MEM_GB > 20:
    BATCH, ACCUM, SEQ, LORA_R, LORA_A = 4, 4, 1024, 16, 32
else:
    BATCH, ACCUM, SEQ, LORA_R, LORA_A = 2, 4, 512, 8, 16
print(f'Config: batch={BATCH} accum={ACCUM} seq={SEQ} lora_r={LORA_R}')
for k in ('RANK','WORLD_SIZE','LOCAL_RANK','LOCAL_WORLD_SIZE','MASTER_ADDR','MASTER_PORT','TORCHELASTIC_RUN_ID','ACCELERATE_TORCH_DEVICE'):
    os.environ.pop(k, None)
os.environ['CUDA_VISIBLE_DEVICES']='0'
os.environ['ACCELERATE_USE_FSDP']='false'
os.environ['ACCELERATE_USE_DEEPSPEED']='false'
cmd = (f'python -m src.training.llm_finetune '
       f'--model_name Qwen/Qwen2.5-7B-Instruct '
       f'--train_file data/llm_seed/train.jsonl --val_file data/llm_seed/val.jsonl '
       f'--output_dir artifacts/qwen25_lora --num_epochs 1 '
       f'--per_device_train_batch_size {BATCH} --gradient_accumulation_steps {ACCUM} '
       f'--learning_rate 2e-4 --lora_r {LORA_R} --lora_alpha {LORA_A} '
       f'--max_seq_length {SEQ}')
print(cmd)
import subprocess
subprocess.run(cmd, shell=True, check=True)

In [ ]:
!python -m src.training.llm_eval \
    --model_name Qwen/Qwen2.5-7B-Instruct \
    --adapter_dir artifacts/qwen25_lora \
    --test_file data/llm_seed/test.jsonl \
    --metrics_path artifacts/metrics_llm.json \
    --max_samples 1000

In [ ]:
import subprocess, json
BUCKET='css-rating2-training-wsd2605'; BASE='css-rating2'
subprocess.run(f'gsutil -m cp -r artifacts/qwen25_lora gs://{BUCKET}/{BASE}/', shell=True)
subprocess.run(f'gsutil cp artifacts/metrics_llm.json gs://{BUCKET}/{BASE}/', shell=True)
print('=== UPLOAD COMPLETE ===')
print(json.dumps(json.load(open('artifacts/metrics_llm.json')), indent=2, ensure_ascii=False))